# GR-RHS-friendly high-dimensional grouped simulation

This notebook rewrites the original `Simulation.ipynb` idea into a simulation design that is better aligned with GR-RHS.

Core changes:

1. Keep the original notebook's idea of high-dimensional correlated features with optional signed correlations and cross-loading.
2. Change the beta-generation mechanism: nonzero coefficients are no longer sprinkled randomly across all features. Instead, active groups are selected first, and concentrated / distributed / mixed signals are generated inside those groups.
3. Allow decoy-like null groups through the correlation structure: they may be correlated with active groups while their true beta values remain zero.
4. Add group-level diagnostics to verify that the simulation truly creates active/null group structure.

Why this is more appropriate for GR-RHS: the model is not only meant to handle high-dimensional correlated features. It is meant to combine group-level shrinkage with within-group local escape. A useful validation dataset should therefore have grouped correlation in `X`, active/null group structure in `beta`, and heterogeneous signals inside active groups.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    import seaborn as sns
except ModuleNotFoundError:
    # If seaborn is unavailable, the diagnostic plots fall back to matplotlib.
    sns = None


## 1. Correlated high-dimensional design matrix

This section keeps the spirit of `Simulation.ipynb`: a signed factor block model generates a high-dimensional feature matrix with complex dependence.

Compared with a simple block-diagonal design, this generator allows:

- mixed positive and negative within-group correlations;
- unequal group sizes;
- correlated group-level factors;
- cross-loading of some features onto other groups;
- a small fraction of orphan/noise variables.

This is closer to the local correlation, global factor variation, and cross-region dependence often seen in real omics data.

In [ ]:
def nearest_correlation(C: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    """Project a symmetric matrix to a numerically valid correlation matrix."""
    C = np.asarray(C, dtype=float)
    C = 0.5 * (C + C.T)
    w, V = np.linalg.eigh(C)
    w = np.maximum(w, eps)
    C_psd = (V * w) @ V.T
    d = np.sqrt(np.maximum(np.diag(C_psd), eps))
    C_corr = C_psd / d[:, None] / d[None, :]
    np.fill_diagonal(C_corr, 1.0)
    return C_corr


def sample_signed_factor_group_design(
    n: int,
    group_sizes: list[int],
    *,
    within_abs_rho: float = 0.65,
    neg_prob_within: float = 0.35,
    loading_mag_jitter: float = 0.25,
    between_factor_rho_mean: float = 0.10,
    between_factor_rho_sd: float = 0.08,
    cross_loading_frac: float = 0.10,
    cross_loading_scale: float = 0.25,
    orphan_frac: float = 0.00,
    random_state: int = 20260428,
) -> tuple[pd.DataFrame, list[np.ndarray], dict]:
    """Generate high-dimensional grouped X with realistic correlation structure.

    MODIFIED/KEPT FROM Simulation.ipynb:
    - Keep the signed factor block model as the X-generation mechanism.
    - Use explicit group_sizes instead of drawing a few very large random clusters from n_clusters.
    - Reason: a GR-RHS benchmark needs explicit groups, for example 50 groups x 10 features.
      This makes the later active/null group structure clear and aligns the simulation with the grouped prior.
    """
    rng = np.random.default_rng(random_state)
    group_sizes = [int(s) for s in group_sizes]
    if min(group_sizes) <= 0:
        raise ValueError("All group sizes must be positive.")

    G = len(group_sizes)
    p = int(sum(group_sizes))
    groups = []
    start = 0
    for size in group_sizes:
        idx = np.arange(start, start + size, dtype=int)
        groups.append(idx)
        start += size

    membership = np.empty(p, dtype=int)
    for g, idx in enumerate(groups):
        membership[idx] = g

    orphan_frac = float(np.clip(orphan_frac, 0.0, 1.0))
    n_orphan = int(round(p * orphan_frac))
    if n_orphan > 0:
        orphan_idx = rng.choice(p, size=n_orphan, replace=False)
        membership[orphan_idx] = -1

    L = np.zeros((p, G), dtype=float)
    for j in range(p):
        g = membership[j]
        if g < 0:
            continue
        sign = -1.0 if rng.random() < neg_prob_within else 1.0
        mag = max(0.05, rng.normal(1.0, loading_mag_jitter))
        L[j, g] = sign * mag

        if G > 1 and rng.random() < cross_loading_frac:
            other = rng.integers(0, G - 1)
            other = other + (other >= g)
            sign2 = -1.0 if rng.random() < 0.5 else 1.0
            L[j, other] = sign2 * mag * cross_loading_scale

    B = np.eye(G, dtype=float)
    if G > 1:
        iu = np.triu_indices(G, 1)
        vals = rng.normal(between_factor_rho_mean, between_factor_rho_sd, size=len(iu[0]))
        vals = np.clip(vals, -0.75, 0.75)
        B[iu] = vals
        B[(iu[1], iu[0])] = vals
    B = nearest_correlation(B)

    within_abs_rho = float(np.clip(within_abs_rho, 1e-6, 0.99))
    noise_var = (1.0 - within_abs_rho) / within_abs_rho
    noise_std = float(np.sqrt(max(noise_var, 1e-8)))

    cholB = np.linalg.cholesky(B)
    F = rng.standard_normal((int(n), G)) @ cholB.T
    E = rng.normal(0.0, noise_std, size=(int(n), p))
    X = F @ L.T + E

    # Scale by population diagonal implied by the factor model.
    LB = L @ B
    diag_S = np.sum(LB * L, axis=1) + noise_std**2
    X = X / np.sqrt(np.maximum(diag_S, 1e-12))[None, :]

    # Final sample standardization keeps downstream methods on the same scale.
    X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, ddof=0, keepdims=True) + 1e-12)

    columns = [f"x{j + 1}" for j in range(p)]
    meta = {
        "n": int(n),
        "p": int(p),
        "n_groups": int(G),
        "group_sizes": group_sizes,
        "membership": membership.tolist(),
        "within_abs_rho_target": float(within_abs_rho),
        "between_factor_rho_mean": float(between_factor_rho_mean),
        "between_factor_rho_sd": float(between_factor_rho_sd),
        "cross_loading_frac": float(cross_loading_frac),
        "cross_loading_scale": float(cross_loading_scale),
        "orphan_frac": float(orphan_frac),
    }
    return pd.DataFrame(X, columns=columns), groups, meta


## 2. GR-RHS-friendly grouped beta generator

This is the most important change relative to the original `Simulation.ipynb`.

In the original notebook, nonzero beta coordinates were sampled randomly from all features. That design mostly tests ordinary high-dimensional sparse regression, not genuinely group-aware shrinkage.

Here the signal-generation mechanism is changed to:

1. Select active groups first.
2. Set beta to zero for every null group.
3. Allow three signal shapes inside active groups:
   - `concentrated`: a few strong coordinates;
   - `distributed`: several weak-to-moderate coordinates;
   - `mixed`: one dominant coordinate plus several weak coordinates.
4. Allow different active groups to have different signal shapes, creating multimode heterogeneous signals.

This is better suited for validating GR-RHS because the model is meant to retain active groups at the group level while allowing heterogeneous within-group escape through local scales.

In [ ]:
def make_grrhs_grouped_beta(
    groups: list[np.ndarray],
    *,
    n_active_groups: int = 5,
    active_groups: list[int] | None = None,
    signal_modes: tuple[str, ...] = ("concentrated", "distributed", "mixed"),
    energy_per_active_group: str = "dirichlet",
    total_signal_energy: float = 1.0,
    support_fraction_range: tuple[float, float] = (0.2, 0.8),
    random_state: int = 20260428,
) -> tuple[pd.Series, dict]:
    """Generate beta with explicit active/null group structure.

    MODIFIED FROM Simulation.ipynb:
    - Original version: causal_idx/nonzero_idx were sampled from all p features.
    - This version: active groups are sampled first, and nonzero beta values are generated only within active groups.

    WHY THIS IS BETTER FOR GR-RHS:
    - The GR-RHS group-specific slab ceiling needs a real active/null group contrast.
    - Concentrated/distributed/mixed within-group shapes test whether local scales can handle heterogeneous signals.
    - Multimode active groups test whether a shrinkage model is too restrictive when active groups do not share one common shape.
    """
    rng = np.random.default_rng(random_state)
    G = len(groups)
    p = int(sum(len(g) for g in groups))
    if active_groups is None:
        active_groups = sorted(rng.choice(G, size=int(n_active_groups), replace=False).tolist())
    else:
        active_groups = sorted(int(g) for g in active_groups)
    if any(g < 0 or g >= G for g in active_groups):
        raise ValueError("active_groups contains an invalid group id.")

    K = len(active_groups)
    if K == 0:
        raise ValueError("At least one active group is required.")

    if energy_per_active_group == "equal":
        group_energy = np.full(K, 1.0 / K)
    elif energy_per_active_group == "dirichlet":
        group_energy = rng.dirichlet(np.full(K, 3.0))
    else:
        raise ValueError("energy_per_active_group must be 'equal' or 'dirichlet'.")
    group_energy = group_energy * float(total_signal_energy)

    beta = np.zeros(p, dtype=float)
    group_records = []
    mode_by_group = {}

    for k, g in enumerate(active_groups):
        idx = np.asarray(groups[g], dtype=int)
        p_g = len(idx)
        mode = str(rng.choice(signal_modes))
        mode_by_group[g] = mode

        if mode == "concentrated":
            # A few strong coordinates: tests whether the model allows local escape inside an active group.
            s = max(1, int(round(0.15 * p_g)))
            active_idx = rng.choice(idx, size=s, replace=False)
            weights = rng.dirichlet(np.full(s, 0.35))
        elif mode == "distributed":
            # Several weak/moderate coordinates: tests whether group-level retention protects distributed signal.
            lo, hi = support_fraction_range
            frac = float(rng.uniform(lo, hi))
            s = max(2, min(p_g, int(round(frac * p_g))))
            active_idx = rng.choice(idx, size=s, replace=False)
            weights = rng.dirichlet(np.full(s, 4.0))
        elif mode == "mixed":
            # One dominant effect plus weak effects: a common shape for many genomic blocks.
            s = max(2, min(p_g, int(round(0.5 * p_g))))
            active_idx = rng.choice(idx, size=s, replace=False)
            weights = np.full(s, 0.0)
            weights[0] = 0.55
            if s > 1:
                weights[1:] = 0.45 * rng.dirichlet(np.full(s - 1, 2.0))
        else:
            raise ValueError(f"Unknown signal mode: {mode}")

        sign = -1.0 if rng.random() < 0.5 else 1.0
        magnitudes = np.sqrt(group_energy[k] * weights)
        beta[active_idx] = sign * magnitudes

        group_records.append(
            {
                "group": int(g),
                "mode": mode,
                "group_size": int(p_g),
                "n_nonzero": int(len(active_idx)),
                "energy": float(group_energy[k]),
                "sign": int(sign),
                "nonzero_indices": active_idx.astype(int).tolist(),
            }
        )

    meta = {
        "active_groups": active_groups,
        "null_groups": [g for g in range(G) if g not in active_groups],
        "mode_by_group": mode_by_group,
        "group_records": group_records,
        "n_nonzero": int(np.sum(beta != 0.0)),
        "total_signal_energy": float(np.sum(beta**2)),
    }
    return pd.Series(beta, index=[f"x{j + 1}" for j in range(p)], name="beta"), meta


## 3. Full simulation wrapper aligned with the low-dimensional benchmark

This wrapper combines grouped correlated `X` with GR-RHS-friendly `beta` and generates a continuous response calibrated to target `R2 = 0.7`.

MODIFIED TO ALIGN WITH `Simulation_second`:

- The original notebook used `signal_to_noise = sd(X beta) / sd(noise)`.
- This notebook uses `target_r2` calibration with default `target_r2 = 0.7`.
- Reason: the low-dimensional main benchmark also calibrates noise by target R2, making the high-dimensional and low-dimensional results more comparable.

Default high-dimensional analogue:

- Low-dimensional benchmark: `n=500, p=50, 5 groups x 10, active groups=3`.
- High-dimensional analogue: `n=200, p=500, 50 groups x 10, active groups=5`.

Both designs share the same core problem: grouped correlated design plus multimode heterogeneous active groups.

In [ ]:
def simulate_grrhs_highdim_grouped_data(
    *,
    n: int = 200,
    group_sizes: list[int] | None = None,
    n_active_groups: int = 5,
    target_r2: float = 0.7,
    random_state: int = 20260428,
    **x_kwargs,
) -> dict:
    """Simulate a high-dimensional dataset designed for GR-RHS validation.

    MODIFIED DESIGN SUMMARY:
    - X: keeps grouped correlation from the original notebook, but defaults are now cleaner.
    - beta: changed to active/null group signal instead of random coordinate sprinkling.
    - y: generated from X @ beta with target-R2 calibrated noise.

    WHY THIS CHANGE MATTERS:
    - The low-dimensional `Simulation_second` benchmark calibrates noise by target R2.
    - Using the same calibration makes this notebook a high-dimensional analogue,
      not a separate stress toy with a different signal scale.
    """
    rng = np.random.default_rng(random_state)
    if group_sizes is None:
        group_sizes = [10] * 50  # p = 500, p > n by default.

    X, groups, x_meta = sample_signed_factor_group_design(
        n=n,
        group_sizes=group_sizes,
        random_state=random_state,
        **x_kwargs,
    )
    beta, beta_meta = make_grrhs_grouped_beta(
        groups,
        n_active_groups=n_active_groups,
        random_state=random_state + 1009,
    )

    linpred = X.to_numpy() @ beta.to_numpy()
    target_r2 = float(target_r2)
    if not (0.0 < target_r2 < 1.0):
        raise ValueError("target_r2 must be in (0, 1).")
    signal_var = float(np.var(linpred, ddof=0))
    noise_sd = float(np.sqrt(signal_var * (1.0 - target_r2) / target_r2))
    y = linpred + rng.normal(0.0, noise_sd, size=int(n))
    y = pd.Series(y, name="y")

    meta = {
        "design": x_meta,
        "signal": beta_meta,
        "target_r2": float(target_r2),
        "noise_sd": noise_sd,
        "empirical_signal_sd": float(np.std(linpred, ddof=0)),
        "empirical_noise_sd": noise_sd,
        "empirical_r2_approx": float(np.var(linpred, ddof=0) / (np.var(linpred, ddof=0) + noise_sd**2)),
    }

    return {"X": X, "y": y, "beta": beta, "groups": groups, "meta": meta}


## 4. Example: high-dimensional analogue of the low-dimensional benchmark

This example is aligned with the low-dimensional `Simulation_second` main benchmark:

- Low-dimensional main benchmark: `n_train = 500`, `p = 50`, `5 groups x 10`, `target R2 = 0.7`.
- High-dimensional analogue: `n = 200`, `p = 500`, `50 groups x 10`, `target R2 = 0.7`.
- Both use active/null groups and concentrated / distributed / mixed within-group heterogeneous signals.

To stay close to the low-dimensional benchmark, cross-loading is disabled by default and between-group factor correlations are centered around 0.2. Cross-loading can be turned back on for a more realistic omics stress test.

In [ ]:
sim = simulate_grrhs_highdim_grouped_data(
    n=200,
    group_sizes=[10] * 50,
    n_active_groups=5,
    target_r2=0.7,
    within_abs_rho=0.80,
    neg_prob_within=0.00,
    between_factor_rho_mean=0.20,
    between_factor_rho_sd=0.03,
    cross_loading_frac=0.00,
    cross_loading_scale=0.00,
    random_state=20260428,
)

X = sim["X"]
y = sim["y"]
beta = sim["beta"]
groups = sim["groups"]
meta = sim["meta"]

print(json.dumps({
    "n": meta["design"]["n"],
    "p": meta["design"]["p"],
    "n_groups": meta["design"]["n_groups"],
    "active_groups": meta["signal"]["active_groups"],
    "n_nonzero_beta": meta["signal"]["n_nonzero"],
    "target_r2": meta["target_r2"],
    "empirical_r2_approx": round(meta["empirical_r2_approx"], 3),
}, indent=2))


## 5. Diagnostics: does the simulation really create grouped signal?

These diagnostics guard against a common failure mode: a dataset can look high-dimensional, correlated, and complex while the true signal has no group structure.

If active groups have positive beta energy and null groups have zero beta energy, the modification succeeded: the dataset can genuinely validate group-aware shrinkage in GR-RHS.

In [ ]:
def group_signal_summary(beta: pd.Series, groups: list[np.ndarray], active_groups: list[int]) -> pd.DataFrame:
    rows = []
    b = beta.to_numpy()
    active_set = set(active_groups)
    for g, idx in enumerate(groups):
        vals = b[idx]
        rows.append(
            {
                "group": g,
                "status": "active" if g in active_set else "null",
                "group_size": len(idx),
                "n_nonzero": int(np.sum(vals != 0.0)),
                "beta_l2": float(np.sqrt(np.sum(vals**2))),
                "beta_energy": float(np.sum(vals**2)),
                "max_abs_beta": float(np.max(np.abs(vals))),
            }
        )
    return pd.DataFrame(rows)


summary = group_signal_summary(beta, groups, meta["signal"]["active_groups"])
summary.sort_values(["status", "beta_energy"], ascending=[True, False]).head(12)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_df = summary.copy()
if sns is not None:
    sns.barplot(data=plot_df, x="group", y="beta_energy", hue="status", dodge=False, ax=axes[0])
else:
    colors = np.where(plot_df["status"].to_numpy() == "active", "tab:red", "tab:blue")
    axes[0].bar(plot_df["group"], plot_df["beta_energy"], color=colors)
axes[0].set_title("True beta energy by group")
axes[0].set_xlabel("Group")
axes[0].set_ylabel("sum(beta^2)")
axes[0].tick_params(axis="x", labelrotation=90, labelsize=6)

nz = beta.to_numpy() != 0
axes[1].stem(np.arange(len(beta)), beta.to_numpy(), markerfmt=".", basefmt=" ")
axes[1].set_title("True beta vector")
axes[1].set_xlabel("Feature index")
axes[1].set_ylabel("beta")

plt.tight_layout()
plt.show()


## 6. Diagnostics: does X have grouped correlation?

This section visualizes a feature-correlation heatmap. Because `p=500`, a full clustermap can be slow, so the example first uses the first 15 groups, i.e. the first 150 variables.

In [ ]:
subset_p = 150
C = X.iloc[:, :subset_p].corr()
if sns is not None:
    sns.clustermap(C, cmap="coolwarm", center=0.0, figsize=(7, 7), xticklabels=False, yticklabels=False)
    plt.show()
else:
    plt.figure(figsize=(7, 6))
    plt.imshow(C, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    plt.colorbar(label="correlation")
    plt.title("Feature correlation heatmap")
    plt.xticks([])
    plt.yticks([])
    plt.tight_layout()
    plt.show()


## 7. Optional export

If this dataset should be passed to a benchmark runner, export `X.csv`, `y.csv`, `beta.csv`, `groups.json`, and `meta.json`.

Export is disabled by default to avoid accidentally creating large files.

In [ ]:
EXPORT = False

if EXPORT:
    out_dir = Path("outputs/history/grrhs_highdim_synthetic_notebook")
    out_dir.mkdir(parents=True, exist_ok=True)
    X.to_csv(out_dir / "X.csv", index=False)
    y.to_csv(out_dir / "y.csv", index=False)
    beta.to_csv(out_dir / "beta.csv")
    with (out_dir / "groups.json").open("w", encoding="utf-8") as f:
        json.dump([g.astype(int).tolist() for g in groups], f, indent=2)
    with (out_dir / "meta.json").open("w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    print(f"Saved to {out_dir}")
